In [ ]:
!pip install scikit-learn
!pip install tensorflow
!pip install torch
!pip install xgboost

In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import vstack
import xgboost as xgb

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_json('C:/Users/007pe/Downloads/tdidf_embeddings.json', lines=True)

In [3]:
df = df.dropna(subset=['Speaker_party_name'])
print("DataFrame after dropping NaN in 'Speaker_party_name':")
print(df)

DataFrame after dropping NaN in 'Speaker_party_name':
                Speaker_party_name  \
0       Centre-right to right-wing   
1            Centre to centre-left   
2       Centre-right to right-wing   
3                      Centre-left   
4       Centre-right to right-wing   
...                            ...   
591683                 Centre-left   
591684  Centre-right to right-wing   
591685                 Centre-left   
591686  Centre-right to right-wing   
591687                 Centre-left   

                                                   tokens  \
0       [government, track, deliver, commitment, intro...   
1       [clear, exit, check, scrap, previous, labour, ...   
2       [indicate, original, answer, track, ensure, ex...   
3       [give, situation, border, calais, home, secret...   
4       [great, deal, work, french, authority, relatio...   
...                                                   ...   
591683  [argument, law, protect, everybody, action, ta...   
5

In [4]:
# print(df)
# df['Speaker_party_name'] = df['Speaker_party_name'].dropna()
# print(df)
# print(df['Speaker_party_name'])
# df2 = df.copy(deep=True)

In [5]:
X = np.vstack(df['embedding'].values)
y = df['Speaker_party_name']

In [6]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)  # Convert labels to integers
num_classes = len(encoder.classes_) 
print(encoder.classes_)
print(num_classes)

['Centre to centre-left' 'Centre-left' 'Centre-left to left-wing'
 'Centre-right to right-wing' 'Cleric' 'Independent' 'Left-wing'
 'Left-wing to far-left' 'Non-partisan' 'Right-wing'
 'Right-wing to far-right']
11


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [8]:
xgb_classifier = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=num_classes,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.01,
    reg_lambda=1.0,
    eval_metric='mlogloss',
    random_state=0,
    n_jobs=3
)

In [9]:
# xgb_classifier2 = xgb.XGBClassifier(n_estimators=100, objective='multi:softmax',
#     num_class=num_classes)

In [10]:
xgb_classifier.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

[0]	validation_0-mlogloss:2.27668
[1]	validation_0-mlogloss:2.17921
[2]	validation_0-mlogloss:2.09474
[3]	validation_0-mlogloss:2.01711
[4]	validation_0-mlogloss:1.95206
[5]	validation_0-mlogloss:1.89101
[6]	validation_0-mlogloss:1.83436
[7]	validation_0-mlogloss:1.78463
[8]	validation_0-mlogloss:1.73773
[9]	validation_0-mlogloss:1.69492
[10]	validation_0-mlogloss:1.65522
[11]	validation_0-mlogloss:1.61834
[12]	validation_0-mlogloss:1.58344
[13]	validation_0-mlogloss:1.55067
[14]	validation_0-mlogloss:1.52039
[15]	validation_0-mlogloss:1.49235
[16]	validation_0-mlogloss:1.46543
[17]	validation_0-mlogloss:1.44070
[18]	validation_0-mlogloss:1.41732
[19]	validation_0-mlogloss:1.39495
[20]	validation_0-mlogloss:1.37421
[21]	validation_0-mlogloss:1.35444
[22]	validation_0-mlogloss:1.33563
[23]	validation_0-mlogloss:1.31766
[24]	validation_0-mlogloss:1.30117
[25]	validation_0-mlogloss:1.28475
[26]	validation_0-mlogloss:1.26910
[27]	validation_0-mlogloss:1.25438
[28]	validation_0-mlogloss:1.2

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=0.1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=3, num_class=11, num_parallel_tree=None, ...)

In [11]:
print("Making predictions...")
predictions = xgb_classifier.predict(X_test)
proba = xgb_classifier.predict_proba(X_test)

# Evaluate
print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Making predictions...

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.09      0.15      6893
           1       0.54      0.49      0.52     28134
           2       0.68      0.27      0.39      6123
           3       0.74      0.92      0.82     69764
           4       0.68      0.19      0.30       456
           5       0.67      0.13      0.21       435
           6       0.81      0.13      0.22       515
           7       0.00      0.00      0.00         3
           8       0.46      0.14      0.22      4155
           9       0.67      0.26      0.38      1607
          10       1.00      0.06      0.12        81

    accuracy                           0.69    118166
   macro avg       0.62      0.24      0.30    118166
weighted avg       0.66      0.69      0.65    118166

Accuracy: 0.6891

Confusion Matrix:
[[  604  2767    96  3209     5     2     1     0   191    18     0]
 [  276 13881   292 13300    13    1